In [1]:
import os
import torch
from torch.utils.data import DataLoader

from dataset import (
    MaridaDatasetLoader,
    find_patch_bases,
    do_img_conf_mask_exist,
)
from preprocessing import (
    normalize_image,
    set_low_conf_for_nan,
    apply_augmentations,
    build_conf_ignore_mask,
    apply_ignore_index_to_target,
    flatten_for_rf,
    compute_dataset_stats,
)

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\sandr\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:

########################
# 1. TRANSFORMS
########################

def train_transform(img, mask, conf):
    """
    Préprocessing appliqué pendant l'entraînement :
    - normalisation
    - NaN/Inf -> conf=3 + img nettoyée
    - augmentations géométriques
    """
    # Normalisation (simple float)
    img = normalize_image(img)

    # Gérer les NaN/Inf -> conf = 3, img nettoyée
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)

    # Augmentations
    img, mask, conf = apply_augmentations(
        img,
        mask,
        conf,
        p_hflip=0.5,
        p_vflip=0.5,
        p_rotate90=0.5,
    )

    return img, mask, conf


def val_transform(img, mask, conf):
    """
    Préprocessing pour validation / test :
    - normalisation
    - NaN/Inf -> conf=3
    PAS d'augmentations.
    """
    img = normalize_image(img)
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)
    return img, mask, conf


In [ ]:
########################
# 2. CONSTRUCTION DES DATASETS
########################

def build_bases(folder):
    """Retourne une liste de bases pour lesquelles img + mask + conf existent."""
    all_bases = find_patch_bases(folder)
    bases = [b for b in all_bases if do_img_conf_mask_exist(folder, b)]
    print(f"{folder} : {len(bases)} patches valides trouvés.")
    return bases


def make_dataloaders(data_root, batch_size=4):
    """
    data_root peut être, par ex. :
        data_root/train
        data_root/val
    Ou alors tu adaptes à ta structure.
    """

    train_folder = os.path.join(data_root, "train")
    val_folder   = os.path.join(data_root, "val")

    # Construire les listes de bases
    train_bases = build_bases(train_folder)
    val_bases   = build_bases(val_folder)

    # Datasets
    train_dataset = MaridaDatasetLoader(
        folder=train_folder,
        bases=train_bases,
        transform=train_transform,
    )

    val_dataset = MaridaDatasetLoader(
        folder=val_folder,
        bases=val_bases,
        transform=val_transform,
    )

    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,   # augmente si tu veux du parallélisme
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    return train_loader, val_loader

In [ ]:
########################
# 3. EXEMPLE: CALCUL DES STATS
########################

def compute_stats_on_whole_dataset(data_root, batch_size=4):
    """
    Exemple de calcul de mean/std globales sur le dataset (sans augmentation).
    On utilise val_transform (sans aug) ou une transform spéciale si tu préfères.
    """
    train_folder = os.path.join(data_root, "train")
    val_folder   = os.path.join(data_root, "val")

    train_bases = build_bases(train_folder)
    val_bases   = build_bases(val_folder)

    # Dataset sans augmentation (val_transform)
    full_dataset = torch.utils.data.ConcatDataset([
        MaridaDatasetLoader(folder=train_folder, bases=train_bases, transform=val_transform),
        MaridaDatasetLoader(folder=val_folder, bases=val_bases, transform=val_transform),
    ])

    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)

    mean, std = compute_dataset_stats(full_loader)
    print("Mean per band:", mean)
    print("Std per band:", std)
    return mean, std

In [ ]:
########################
# 4. EXEMPLE: BOUCLE D'ENTRAÎNEMENT (pseudo-code)
########################

def train_one_epoch(model, train_loader, optimizer, criterion, device="cpu"):
    model.train()

    for batch_idx, (imgs, masks, confs) in enumerate(train_loader):
        # imgs : (B, C, H, W)
        # masks: (B, H, W)
        # confs: (B, H, W)

        imgs  = imgs.to(device)
        masks = masks.to(device)
        confs = confs.to(device)

        # 1) Construire le target avec ignore_index basé sur la confidence
        targets_for_loss = []
        for b in range(imgs.shape[0]):
            conf_b = confs[b]   # (H, W)
            mask_b = masks[b]   # (H, W)

            ignore_mask = build_conf_ignore_mask(conf_b, threshold=2)
            target_mod  = apply_ignore_index_to_target(
                mask_b,
                ignore_mask,
                ignore_index=-100,
            )
            targets_for_loss.append(target_mod)

        targets_for_loss = torch.stack(targets_for_loss, dim=0)  # (B, H, W)

        # 2) Forward
        optimizer.zero_grad()
        logits = model(imgs)   # (B, num_classes, H, W) par exemple

        # 3) Loss (par ex. CrossEntropy2D avec ignore_index=-100)
        loss = criterion(logits, targets_for_loss)

        # 4) Backprop
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}, loss = {loss.item():.4f}")


########################
# 5. EXEMPLE: DATASET FLATTEN POUR RANDOM FOREST
########################

def build_rf_dataset(data_root):
    """
    Construit X, y pour RandomForest à partir de tous les patches
    (train + val, à adapter selon tes besoins).
    On utilise val_transform (pas d'augmentation aléatoire).
    """
    train_folder = os.path.join(data_root, "train")
    val_folder   = os.path.join(data_root, "val")

    train_bases = build_bases(train_folder)
    val_bases   = build_bases(val_folder)

    train_dataset = MaridaDatasetLoader(
        folder=train_folder,
        bases=train_bases,
        transform=val_transform,
    )
    val_dataset = MaridaDatasetLoader(
        folder=val_folder,
        bases=val_bases,
        transform=val_transform,
    )

    rf_dataset = torch.utils.data.ConcatDataset([train_dataset, val_dataset])
    rf_loader  = DataLoader(rf_dataset, batch_size=1, shuffle=False)

    all_X = []
    all_y = []

    for img, mask, conf in rf_loader:
        img  = img.squeeze(0)   # (C, H, W)
        mask = mask.squeeze(0)  # (H, W)
        conf = conf.squeeze(0)  # (H, W)

        X_rf, y_rf = flatten_for_rf(img, mask, conf, conf_threshold=2)
        all_X.append(X_rf)
        all_y.append(y_rf)

    X_all = torch.cat(all_X, dim=0)
    Y_all = torch.cat(all_y, dim=0)

    print("RF dataset : X =", X_all.shape, ", y =", Y_all.shape)
    return X_all, Y_all

In [ ]:
data_root = "/chemin/vers/tes/donnees"  # adapte-moi ça

    # 1) Créer les dataloaders pour deep learning
train_loader, val_loader = make_dataloaders(data_root, batch_size=4)

    # 2) Exemple: calculer les stats si tu veux
    # mean, std = compute_stats_on_whole_dataset(data_root)

    # 3) Exemple: RF dataset
    # X_all, Y_all = build_rf_dataset(data_root)
    # -> ensuite tu peux faire du scikit-learn sur X_all, Y_all

    # 4) Exemple: training modèle deep (pseudo-code)
    # from model import MyUNet
    # model = MyUNet(num_classes=15).to("cuda")
    # optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    # criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
    #
    # for epoch in range(10):
    #     print(f"\nEpoch {epoch}")
    #     train_one_epoch(model, train_loader, optimizer, criterion, device="cuda")

print("Main terminé (adapter les parties modèle / RF selon ton besoin).")
